### Structured Output

Models can be requested to provide their response in a format matching to given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing.

### Pydantic

Pydantic models provide the richest feature set with field validation, description and nested structures.

In [2]:
import os 
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] =  os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x7c7cb4d7ef90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7c7cb4d7fcb0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="Title of the movie")
    year: int = Field(description="Release year of the movie")
    director: str = Field(description="Director of the movie")
    ratings: float = Field(description="Ratings of the movie")
    


In [4]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x7c7cb4d7ef90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7c7cb4d7fcb0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'Title of the movie', 'type': 'string'}, 'year': {'description': 'Release year of the movie', 'type': 'integer'}, 'director': {'description': 'Director of the movie', 'type': 'string'}, 'ratings': {'description': 'Ratings of the movie', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'ratings'], 'type'

In [5]:
model.invoke("Provide the details of {Inception} movie")

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Inception. Let me start by recalling what I know. The director is Christopher Nolan, right? The main cast includes Leonardo DiCaprio, Joseph Gordon-Levitt, and others. The movie is about dreams and something called inception, which I think is about planting an idea in someone\'s mind. \n\nThe setting is probably a near-future or present-day world. The main character, Dom Cobb, is a thief who steals information by entering people\'s dreams. But instead of doing that, he\'s asked to plant an idea. The concept is pretty complex, involving layers of dreams. There\'s something about time slowing down depending on the level of the dream. Also, there\'s a spinning top that Cobb uses to check if he\'s in a dream, which is a totem.\n\nThe plot might involve a mission where they go into multiple layers of dreams, each deeper than the last. There are other characters like Arthur, who is Cobb\'s right-hand man, and Aria

In [6]:
model_with_structure.invoke("Provide the details of {Inception} movie")

Movie(title='Inception', year=2010, director='Christopher Nolan', ratings=8.8)

### Message output alongside parsed structure

Parsed structure (include_raw = True)

In [7]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details"""
    title: str = Field(description="Title of the movie")
    year: int = Field(description="Release year of the movie")
    director: str = Field(description="Director of the movie")
    ratings: float = Field(description="Ratings of the movie")
    
model_with_structure = model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke("Provide the details of {Inception} movie")
response


{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the details of the movie "Inception". Let me check the available functions. There\'s a function called Movie with parameters title, year, director, and ratings. All these are required. I need to make sure I have all the correct information for Inception. The title is obviously "Inception". The director is Christopher Nolan. The release year was 2010. As for ratings, I think it\'s around 8.8 on IMDb. I should structure the function call with these details. Let me double-check the parameters to ensure I\'m not missing anything. Yep, all required fields are covered. Alright, time to format the JSON response correctly.\n', 'tool_calls': [{'id': 'dr6x8xafc', 'function': {'arguments': '{"director":"Christopher Nolan","ratings":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 190, 'prompt_tokens': 226, 'total

### Nested structure

In [8]:
from pydantic import BaseModel, Field
class Actor( BaseModel):
    name : str
    role : str

class MovieDetails(BaseModel):
    title: str
    year : int
    cast : list[Actor]    # nested structure
    genres: list[str]
    budget: float | None = Field (None,description ="Budget in Million dollars")


In [9]:
model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide the details about Inception movie")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Bane')], genres=['Action', 'Science Fiction', 'Thriller'], budget=160.0)

### TypedDict

TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation

In [10]:
from typing import TypedDict
class Actor( TypedDict):
    name : str
    role : str

class MovieDetails(TypedDict):
    title: str
    year : int
    cast : list[Actor]    # nested structure
    genres: list[str]
    budget: float | None = Field (None,description ="Budget in Million dollars")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide the details about Inception movie")
response


{'budget': 160000000,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Ellen Page', 'role': 'Ariadne'},
  {'name': 'Tom Hardy', 'role': 'Balthazar'}],
 'genres': ['Science Fiction', 'Action'],
 'title': 'Inception',
 'year': 2010}

### Dataclasses

In [15]:
from dataclasses import dataclass  
from langchain.agents import create_agent

@dataclass
class Contactinfo:
    """ Contact details of person"""
    name : str
    email : str 
    contact : str 

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format = Contactinfo
)

response = agent.invoke({
    "messages" : [{"role":"user", "content": "Extract the contact info from: John, abcd@gmail.com, 9893257097"}]
})
response["structured_response"]

Contactinfo(name='John', email='abcd@gmail.com', contact='9893257097')